# LangChain RAG Pipeline — Hadith (Local)

Hadith RAG over `data/dorar_hadith_full_batch_2.csv` (or your local CSV).  
Each section is one pipeline stage; change **`CONFIG`** to swap models, chunking, or retrieval.

**Run order:** Install → Config → Imports → normalize_arabic → Load → Chunk → Embeddings → Vector Store → BM25 → Hybrid Retriever → LLM → Prompt → Chain → Query → Evaluation

**Requirements:** `pip install -r requirements.txt`  (copy `.env.example` → `.env` for API keys)


## TODO:
- show similirty for each doc => to set a threshold
- merge George APIs into code
- front & backend & docker
- searcj for system prompt حلو
- run script to try all alpha [0 - 1] with diffrnent K and save them as a rerfrence

## Install

In [1]:
!python -m pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python -m pip install langchain_huggingface langchain_chroma rank_bm25 nltk rouge-score --quiet


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Config

In [3]:
from pathlib import Path

CONFIG = {
  "paths": {
    "data_csv": Path("data/semantic_clustered_hadiths_per_sharh.csv"),
    "chroma_dir": Path("chroma_db"),
    "collection_name": "hadith_rag",
  },
  "data": {
    "max_rows": None,          # None = full CSV. Start small while experimenting.
    "text_columns": ["sharh"],
    "metadata_columns":
        ['page_id', 'url', 'categories', 'sharh', 'hadith', 'rawy',
       'mohadth', 'source', 'page', 'hokm', 'takhrij'],
  },
  "chunking": {
    "chunk_size": 450, #TODO : tune this parameter for best performance
    "chunk_overlap": 40,
  },
  "embeddings": {
    "provider": "huggingface",   # "huggingface" | "openai"
    "model_name": "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
    "openai_model": "text-embedding-3-small",
  },
  "vector_store": {
    "persist": True,
    "reset_on_build": False,   # True = delete collection and re-index
  },
  "retriever": {
    "search_type": "similarity",   # "similarity" | "mmr" | "hybrid"
    "k": 5,
    "fetch_k": 25,             # used when search_type == "mmr"
    "lambda_mult": 0.5,
    # Hybrid weights (dense + BM25); must sum to 1.0
    "hybrid_alpha": 1,       # 0 = BM25 only, 1 = dense only
  },
  "llm": {
    "provider": "groq",        # "openai" | "ollama" | "groq" | "huggingface_local" | "fanar"
    "openai_model": "gpt-4o-mini",
    "ollama_model": "llama3.2",
    "groq_model": "openai/gpt-oss-20b",
    "huggingface_model": "silma-ai/SILMA-Kashif-2B-Instruct-v1.0",
    "temperature": 0.1,
  },
  "prompt": {
    "language": "ar",
    "system_role": (
        "انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمة"
      "اذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة"
    ),
  },
}

PROJECT_ROOT = Path(".").resolve()
if not CONFIG["paths"]["data_csv"].is_absolute():
    CONFIG["paths"]["data_csv"] = PROJECT_ROOT / CONFIG["paths"]["data_csv"]
if not CONFIG["paths"]["chroma_dir"].is_absolute():
    CONFIG["paths"]["chroma_dir"] = PROJECT_ROOT / CONFIG["paths"]["chroma_dir"]


## Imports

In [4]:
import os
import re
import ast
import shutil
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langsmith import traceable

load_dotenv()

def cfg(*keys: str) -> Any:
    """Read nested CONFIG values, e.g. cfg('retriever', 'k')."""
    node = CONFIG
    for key in keys:
        node = node[key]
    return node

def _get_api_key(key_name: str) -> str | None:
    """Read from .env / environment (no Kaggle dependency for local runs)."""
    return os.environ.get(key_name)

c:\Users\moham\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## normalize_arabic

Strips tashkeel, normalises أإآ→ا, ة→ه, ى→ي, removes BOM and non-word chars.  
Used for both retrieval eval and BLEU/ROUGE scoring; fixes diacritical mismatches between scraped data and eval CSV.


In [5]:
ARABIC_TASHKEEL = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
NON_WORD = re.compile(r"[^\w\s\u0600-\u06FF]+", re.UNICODE)


def normalize_arabic(text: str) -> str:
    """ArithmeticError normalization for Arabic text: remove diacritics, normalize letters, and remove non-word characters."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = str(text)
    text = ARABIC_TASHKEEL.sub("", text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = NON_WORD.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_set(text: str) -> set[str]:
    return {t for t in normalize_arabic(text).split() if len(t) > 1}


def token_overlap_ratio(a: str, b: str) -> float:
    ta, tb = token_set(a), token_set(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta)


def sequence_matcher(a: str, b: str) -> float:
    from difflib import SequenceMatcher
    return SequenceMatcher(None, normalize_arabic(a), normalize_arabic(b)).ratio()


## Data Loading

Each hadith gets its own aligned record (`rawy`, `mohadth`, `hokm`, `takhrij` all at matching indices).


In [6]:
HADITH_LEVEL_LIST_COLUMNS = cfg("data", "metadata_columns")
EMBED_COLUMN = cfg("data", "text_columns")[0]


def load_hadith_documents(csv_path=None, max_rows: int | None = None) -> list[Document]:
    csv_path = Path(csv_path or cfg("paths", "data_csv"))
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    if max_rows:
        df = df.head(max_rows)
    df = df[df[EMBED_COLUMN].notna()]

    docs: list[Document] = []

    for sharh_text, group in df.groupby(EMBED_COLUMN, sort=False):
        metadata = {
            col: group[col].fillna("").tolist() for col in HADITH_LEVEL_LIST_COLUMNS
        }

        docs.append(
            Document(
                page_content=str(sharh_text).strip(),
                metadata=metadata,
            )
        )
    print(f"Loaded {len(docs)} documents from {csv_path.name}")
    return docs


raw_documents = load_hadith_documents(
    cfg("paths", "data_csv"),
    max_rows=cfg("data", "max_rows"),
)
print(raw_documents[0].page_content[:200] if raw_documents else "No documents")


Loaded 14691 documents from semantic_clustered_hadiths_per_sharh.csv
"اللهم لك الحمد أنت كسوتنيه"، أي: أنت الذي رزقتني به من غير حول مني ولا قوة، "أسألك من خيره وخير ما صنع له"، أي: أعني على أن أستعمله في طاعتك وعبادتك، ويكون عونا لي فيهما، "وأعوذ بك من شره وشر ما صنع 


## Chunking — Token-based Parent-Child

Replaces `RecursiveCharacterTextSplitter` with `AutoTokenizer` splits into token-sized chunks.  
Keeps a `PARENT_STORE` so retrieval fetches full parent docs.  
More accurate for Arabic text than character-count chunking.


In [7]:
from transformers import AutoTokenizer

PARENT_STORE: list[Document] = []


def split_by_tokens(tokenizer, text: str, chunk_size: int = 350, chunk_overlap: int = 32) -> list[str]:
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    step = max(chunk_size - chunk_overlap, 1)
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i: i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        if chunk_text.strip():
            chunks.append(chunk_text.strip())
    return chunks


def build_child_chunks(raw_docs: list[Document], chunk_size: int = 350, chunk_overlap: int = 32) -> list[Document]:
    """Split each parent document into token-sized child chunks."""
    global PARENT_STORE
    PARENT_STORE = []

    child_docs: list[Document] = []
    model_name = cfg("embeddings", "model_name")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    for parent_id, parent_doc in enumerate(raw_docs):
        PARENT_STORE.append(parent_doc)

        for local_idx, chunk_text in enumerate(split_by_tokens(tokenizer, parent_doc.page_content, chunk_size, chunk_overlap)):
            global_idx = len(child_docs)
            child_meta = {
                "idx": global_idx,
                "parent_id": parent_id,
                "chunk_index": local_idx,
            } | parent_doc.metadata
            child_docs.append(Document(page_content=chunk_text, metadata=child_meta))

    print(f"Parents: {len(PARENT_STORE)} | Child chunks: {len(child_docs)}")
    return child_docs


chunks = build_child_chunks(
    raw_documents,
    chunk_size=cfg("chunking", "chunk_size"),
    chunk_overlap=cfg("chunking", "chunk_overlap"),
)
print(f"Child chunks to index: {len(chunks)}")

# sanity check: idx in metadata must match position in the list
assert all(c.metadata["idx"] == i for i, c in enumerate(chunks)), "chunk idx/position mismatch!"

Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


Parents: 14691 | Child chunks: 19141
Child chunks to index: 19141


## Embedding Model

In [8]:
def build_embeddings():
    provider = cfg("embeddings", "provider")
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=cfg("embeddings", "openai_model"))
    if provider == "huggingface":
        from langchain_huggingface import HuggingFaceEmbeddings
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        return HuggingFaceEmbeddings(
            model_name=cfg("embeddings", "model_name"),
            model_kwargs={"device": device},
            encode_kwargs={"batch_size": 256},
        )
    raise ValueError(f"Unknown embeddings provider: {provider}")


embeddings = build_embeddings()
print(f"Embeddings ready: {cfg('embeddings', 'provider')} / {cfg('embeddings', 'model_name')}")


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings ready: huggingface / Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2


## Vector Store Qdrant

In [9]:
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

QDRANT_URL = _get_api_key("QDRANT_URL")
QDRANT_API_KEY = _get_api_key("QDRANT_API_KEY")
COLLECTION_NAME = cfg("paths", "collection_name")
embedding_dim = 768
client = QdrantClient(
    host="localhost",
    port=6333,
    timeout=600,
    prefer_grpc=True
)


# Create collection if needed
collections = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME not in collections:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE,
        ),
    )

# Check whether already indexed
collection_info = client.get_collection(COLLECTION_NAME)
existing_count = collection_info.points_count or 0
expected_count = len(chunks)

if existing_count == expected_count:
    print(f"Collection already contains {existing_count} points. Skipping indexing.")

else:
    if existing_count > 0:
        print(
            f"Collection contains {existing_count} points "
            f"(expected {expected_count}). Recreating collection..."
        )

        client.delete_collection(COLLECTION_NAME)

        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(
                size=embedding_dim,
                distance=Distance.COSINE,
            ),
        )
    device = "cuda" if torch.cuda.is_available() else "cpu"

    st_model = SentenceTransformer(
        cfg("embeddings", "model_name"),
        device=device,
    )

    if device == "cuda":
        st_model.half()
        torch.backends.cudnn.benchmark = True

    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]

    print(f"Generating embeddings for {len(texts)} chunks on {device}...")

    try:
        embs
    except NameError:
        embs = st_model.encode(
            texts,
            batch_size=256,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

    print("Uploading to Qdrant...")

    batch_size = 100

    for start in tqdm(range(0, len(texts), batch_size)):
        end = min(start + batch_size, len(texts))

        points = [
            PointStruct(
                id=chunks[i].metadata["idx"],
                vector=embs[i].tolist(),
                payload={
                    "page_content": texts[i],
                    "metadata": metadatas[i],   # <-- nest it
                },
            )
            for i in range(start, end)
        ]

        client.upload_points(
            collection_name=COLLECTION_NAME,
            points=points,
            batch_size=64,
            parallel=1,
            max_retries=100,
            wait=True,
        )

    print(f"Indexed {len(texts)} chunks.")

Collection already contains 19141 points. Skipping indexing.


In [10]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

## Chroma (old version)

In [11]:
# import torch
# from langchain_chroma import Chroma
# from sentence_transformers import SentenceTransformer
# from tqdm import tqdm

# chroma_dir = cfg("paths", "chroma_dir")
# collection_name = cfg("paths", "collection_name")

# if cfg("vector_store", "reset_on_build") and chroma_dir.exists():
#     shutil.rmtree(chroma_dir)
#     print(f"Removed {chroma_dir}")

# # cosine similarity — correct for normalised Matryoshka embeddings
# vectorstore = Chroma(
#     collection_name=collection_name,
#     embedding_function=embeddings,
#     persist_directory=str(chroma_dir) if cfg("vector_store", "persist") else None,
#     collection_metadata={"hnsw:space": "cosine"},
# )


# def _collection_has_vectors(vs: Chroma) -> bool:
#     return bool(vs.get(limit=1).get("ids"))


# if not _collection_has_vectors(vectorstore):
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     st_model = SentenceTransformer(cfg("embeddings", "model_name"), device=device)
#     if device == "cuda":
#         st_model.half()
#         torch.backends.cudnn.benchmark = True

#     texts     = [c.page_content for c in chunks]
#     metadatas = [c.metadata for c in chunks]
#     ids       = [str(i) for i in range(len(chunks))]

#     print(f"Generating embeddings for {len(texts)} chunks on {st_model.device}...")
#     embs = st_model.encode(
#         texts,
#         batch_size=64,
#         show_progress_bar=True,
#         convert_to_numpy=True,
#         normalize_embeddings=True,
#     )

#     print("Writing to Chroma...")
#     collection = vectorstore._collection
#     insert_batch_size = 5000
#     for start in tqdm(range(0, len(texts), insert_batch_size)):
#         end = start + insert_batch_size
#         collection.add(
#             ids=ids[start:end],
#             documents=texts[start:end],
#             metadatas=metadatas[start:end],
#             embeddings=embs[start:end].tolist(),
#         )
#     print(f"Indexed {len(chunks)} chunks into '{collection_name}'")
# else:
#     n = len(vectorstore.get().get("ids", []))
#     print(f"Using existing index: {n} vectors in '{collection_name}'")

# vectorstore


## BM25 Index

Built over the same child chunks as the dense index.  
Used by the hybrid retriever to blend lexical and semantic scores.


In [12]:
from rank_bm25 import BM25Okapi

_bm25_corpus = [normalize_arabic(c.page_content).split() for c in chunks]
_bm25_index = BM25Okapi(_bm25_corpus)

print(f"BM25 index: {len(_bm25_corpus)} chunks")

BM25 index: 19141 chunks


## Retrieval

### retrieve_filtered — per-hadith metadata filtering

Fetches child chunks → looks up full parent in `PARENT_STORE` → filters hadiths per-record.  
Exact match for controlled vocab (`rawy`, `hokm`, `mohadth`, `source`) vs substring for free-text (`categories`).  
Returns structured `RetrievedResult` objects.

### Hybrid retrieval (BM25 + dense)

When `search_type == "hybrid"`, dense similarity scores and BM25 scores are combined via:  
`score = alpha × dense + (1 − alpha) × bm25`  
`hybrid_alpha=0.7` in CONFIG means 70 % semantic, 30 % lexical.


## Retrieval

Qdrant dense search (+ optional BM25 hybrid via `CONFIG["retriever"]["search_type"]`).  
Fetches child chunks → assembles parent hadiths from `PARENT_STORE`.

In [13]:
import ast
import pandas as pd
from dataclasses import dataclass, field
from langchain_core.documents import Document

EXACT_MATCH_COLUMNS = {"rawy", "hokm", "mohadth", "source"}


@dataclass
class RetrievedResult:
    sharh: str
    hadiths: list[str]
    list_meta: dict = field(default_factory=dict)
    scalar_meta: dict = field(default_factory=dict)
    score: float | None = None
    dense_score: float | None = None   # raw cosine similarity from Qdrant, if available
    bm25_score: float | None = None    # raw BM25 score, if available


# ---------- metadata parsing (unchanged) ----------

def _safe_parse_list(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(v) if pd.notna(v) else "" for v in raw]
    if isinstance(raw, str) and raw.startswith("["):
        try:
            parsed = ast.literal_eval(raw)
            return [str(v) if pd.notna(v) else "" for v in parsed]
        except (ValueError, SyntaxError):
            pass
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return [""]
    return [str(raw)]


def _build_row_records(meta: dict) -> list[dict]:
    parsed = {col: _safe_parse_list(meta.get(col, [])) for col in HADITH_LEVEL_LIST_COLUMNS}
    n_rows = len(parsed.get("hadith", []))
    return [
        {col: parsed[col][i] if i < len(parsed[col]) else "" for col in HADITH_LEVEL_LIST_COLUMNS}
        for i in range(n_rows)
    ]


def _matches_filters(rec: dict, norm_filters: dict[str, list[str]]) -> bool:
    for col, allowed in norm_filters.items():
        val = str(rec.get(col, "")).lower()
        if col in EXACT_MATCH_COLUMNS:
            if val not in allowed:
                return False
        elif not any(a in val for a in allowed):
            return False
    return True


def _filter_row_records(records: list[dict], norm_filters: dict[str, list[str]]) -> list[dict]:
    if not norm_filters:
        return records
    return [r for r in records if _matches_filters(r, norm_filters)]


def _records_to_list_meta(records: list[dict]) -> dict[str, list[str]]:
    if not records:
        return {col: [] for col in HADITH_LEVEL_LIST_COLUMNS}
    return {col: [rec.get(col, "") for rec in records] for col in HADITH_LEVEL_LIST_COLUMNS}


# ---------- retrieval ----------

def _resolve_chunk_idx(doc: Document) -> int | None:
    idx = doc.metadata.get("idx")
    return int(idx) if idx is not None else None


def _dense_search(query: str, k: int) -> list[tuple[int, float]]:
    """(idx, score) pairs, best first. Score = Qdrant's raw similarity (cosine)."""
    pairs = vectorstore.similarity_search_with_score(query, k=k)
    out = []
    for doc, score in pairs:
        idx = _resolve_chunk_idx(doc)
        if idx is not None:
            out.append((idx, float(score)))
    return out


def _bm25_search(query: str, k: int) -> list[tuple[int, float]]:
    """(idx, score) pairs, best first. Score = raw BM25 score (unbounded)."""
    tokens = normalize_arabic(query).split()
    scores = _bm25_index.get_scores(tokens)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, float(scores[i])) for i in ranked[:k] if scores[i] > 0]


def _reciprocal_rank_fusion(
    rank_lists: list[tuple[list[int], float]],
    k: int,
    rrf_k: int = 60,
) -> list[int]:
    scores: dict[int, float] = {}
    for ranked, weight in rank_lists:
        for rank, idx in enumerate(ranked):
            scores[idx] = scores.get(idx, 0.0) + weight / (rrf_k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)[:k]


def _hybrid_search(
    query: str, k: int, alpha: float
) -> tuple[list[int], dict[int, float], dict[int, float]]:
    dense = _dense_search(query, k)
    bm25 = _bm25_search(query, k)
    dense_idxs = [i for i, _ in dense]
    bm25_idxs = [i for i, _ in bm25]
    fused = _reciprocal_rank_fusion([(dense_idxs, alpha), (bm25_idxs, 1 - alpha)], k=k)
    return fused, dict(dense), dict(bm25)


def _search_candidates(
    query: str, k_fetch: int
) -> tuple[list[Document], dict[int, float], dict[int, float]]:
    """Returns (docs, dense_scores_by_idx, bm25_scores_by_idx)."""
    if cfg("retriever", "search_type") == "hybrid":
        fused_idxs, dense_scores, bm25_scores = _hybrid_search(
            query, k_fetch, cfg("retriever", "hybrid_alpha")
        )
        docs = [chunks[i] for i in fused_idxs]
        return docs, dense_scores, bm25_scores

    dense = _dense_search(query, k_fetch)
    dense_scores = dict(dense)
    docs = [chunks[i] for i, _ in dense]
    return docs, dense_scores, {}


def print_results(results: list[RetrievedResult], max_hadith_preview: int = 120) -> None:
    for i, r in enumerate(results, 1):
        print("─" * 60)
        rawy = [v if v.strip() else "Unknown" for v in r.list_meta.get("rawy", [])]
        print(f"[{i}] rawy={rawy}  dense={r.dense_score}  bm25={r.bm25_score}")
        print(f"    sharh: {r.sharh[:150].replace(chr(10), ' ')}…")
        for j, h in enumerate(r.hadiths):
            label = rawy[j] if j < len(rawy) else "Unknown"
            preview = h[:max_hadith_preview] if max_hadith_preview else h
            print(f"      [{j+1}] {label!r}: {preview}")
    print("─" * 60)

@traceable(name="retreive method")
def retrieve(
    query: str,
    k: int | None = None,
    filters: dict | None = None,
    min_dense_score: float | None = None,
    min_bm25_score: float | None = None,
) -> list[RetrievedResult]:
    k = k or cfg("retriever", "k")
    k_fetch = k * 10
    norm_filters = {
        col: [str(v).lower() for v in (val if isinstance(val, list) else [val])]
        for col, val in (filters or {}).items()
    }

    docs, dense_scores, bm25_scores = _search_candidates(query, k_fetch)

    results: list[RetrievedResult] = []
    seen_parents: set = set()

    for doc in docs:
        idx = _resolve_chunk_idx(doc)
        dense_score = dense_scores.get(idx) if idx is not None else None
        bm25_score = bm25_scores.get(idx) if idx is not None else None

        # threshold filter — set once you've inspected the score distribution
        if min_dense_score is not None and (dense_score is None or dense_score < min_dense_score):
            continue
        if min_bm25_score is not None and (bm25_score is None or bm25_score < min_bm25_score):
            continue

        parent_id = doc.metadata.get("parent_id")
        if parent_id is None or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)

        parent_doc = PARENT_STORE[parent_id]
        records = _build_row_records(parent_doc.metadata)
        kept = _filter_row_records(records, norm_filters)
        if not kept:
            continue

        list_meta = _records_to_list_meta(kept)
        hadiths = list_meta.pop("hadith", [])
        if not hadiths:
            continue

        results.append(RetrievedResult(
            sharh=parent_doc.page_content,
            hadiths=hadiths,
            list_meta=list_meta,
            dense_score=dense_score,
            bm25_score=bm25_score,
        ))
        if len(results) >= k:
            break

    return results


print(f"Retriever ready ✓  (search_type={cfg('retriever', 'search_type')})")

Retriever ready ✓  (search_type=similarity)


In [14]:
# import ast
# import pandas as pd
# from dataclasses import dataclass, field
# from langchain_core.documents import Document

# EXACT_MATCH_COLUMNS = {"rawy", "hokm", "mohadth", "source"}


# @dataclass
# class RetrievedResult:
#     sharh: str
#     hadiths: list[str]
#     list_meta: dict = field(default_factory=dict)
#     scalar_meta: dict = field(default_factory=dict)
#     score: float | None = None


# # ---------- metadata parsing ----------

# def _safe_parse_list(raw) -> list[str]:
#     """Normalize a metadata value into a list[str]."""
#     if isinstance(raw, list):
#         return [str(v) if pd.notna(v) else "" for v in raw]
#     if isinstance(raw, str) and raw.startswith("["):
#         try:
#             parsed = ast.literal_eval(raw)
#             return [str(v) if pd.notna(v) else "" for v in parsed]
#         except (ValueError, SyntaxError):
#             pass
#     if raw is None or (isinstance(raw, float) and pd.isna(raw)):
#         return [""]
#     return [str(raw)]


# def _build_row_records(meta: dict) -> list[dict]:
#     parsed = {col: _safe_parse_list(meta.get(col, [])) for col in HADITH_LEVEL_LIST_COLUMNS}
#     n_rows = len(parsed.get("hadith", []))
#     return [
#         {col: parsed[col][i] if i < len(parsed[col]) else "" for col in HADITH_LEVEL_LIST_COLUMNS}
#         for i in range(n_rows)
#     ]


# def _matches_filters(rec: dict, norm_filters: dict[str, list[str]]) -> bool:
#     """Single source of truth for whether a row satisfies the requested filters."""
#     for col, allowed in norm_filters.items():
#         val = str(rec.get(col, "")).lower()
#         if col in EXACT_MATCH_COLUMNS:
#             if val not in allowed:
#                 return False
#         elif not any(a in val for a in allowed):
#             return False
#     return True


# def _filter_row_records(records: list[dict], norm_filters: dict[str, list[str]]) -> list[dict]:
#     if not norm_filters:
#         return records
#     return [r for r in records if _matches_filters(r, norm_filters)]


# def _records_to_list_meta(records: list[dict]) -> dict[str, list[str]]:
#     if not records:
#         return {col: [] for col in HADITH_LEVEL_LIST_COLUMNS}
#     return {col: [rec.get(col, "") for rec in records] for col in HADITH_LEVEL_LIST_COLUMNS}


# # ---------- retrieval ----------

# def _resolve_chunk_idx(doc: Document) -> int | None:
#     idx = doc.metadata.get("idx")
#     return int(idx) if idx is not None else None


# def _dense_search(query: str, k: int) -> list[int]:
#     """Chunk indices ranked by dense similarity, best first."""
#     pairs = vectorstore.similarity_search_with_score(query, k=k)
#     return [idx for doc, _ in pairs if (idx := _resolve_chunk_idx(doc)) is not None]


# def _bm25_search(query: str, k: int) -> list[int]:
#     """Chunk indices ranked by BM25 score, best first."""
#     tokens = normalize_arabic(query).split()
#     scores = _bm25_index.get_scores(tokens)
#     ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
#     return [i for i in ranked[:k] if scores[i] > 0]


# def _reciprocal_rank_fusion(
#     rank_lists: list[tuple[list[int], float]],
#     k: int,
#     rrf_k: int = 60,
# ) -> list[int]:
#     """Weighted RRF: scale-invariant, avoids mixing BM25/cosine score ranges."""
#     scores: dict[int, float] = {}
#     for ranked, weight in rank_lists:
#         for rank, idx in enumerate(ranked):
#             scores[idx] = scores.get(idx, 0.0) + weight / (rrf_k + rank + 1)
#     return sorted(scores, key=scores.get, reverse=True)[:k]


# def _hybrid_search(query: str, k: int, alpha: float) -> list[Document]:
#     dense_idxs = _dense_search(query, k)
#     bm25_idxs = _bm25_search(query, k)
#     fused = _reciprocal_rank_fusion([(dense_idxs, alpha), (bm25_idxs, 1 - alpha)], k=k)
#     return [chunks[i] for i in fused]


# def _search_candidates(query: str, k_fetch: int) -> list[Document]:
#     if cfg("retriever", "search_type") == "hybrid":
#         return _hybrid_search(query, k_fetch, cfg("retriever", "hybrid_alpha"))
#     return vectorstore.similarity_search(query, k=k_fetch)


# def print_results(results: list[RetrievedResult], max_hadith_preview: int = 120) -> None:
#     for i, r in enumerate(results, 1):
#         print("─" * 60)
#         rawy = [v if v.strip() else "Unknown" for v in r.list_meta.get("rawy", [])]
#         print(f"[{i}] rawy={rawy}")
#         print(f"    sharh: {r.sharh[:150].replace(chr(10), ' ')}…")
#         for j, h in enumerate(r.hadiths):
#             label = rawy[j] if j < len(rawy) else "Unknown"
#             preview = h[:max_hadith_preview] if max_hadith_preview else h
#             print(f"      [{j+1}] {label!r}: {preview}")
#     print("─" * 60)


# @traceable(name="retreive method")
# def retrieve(query: str, k: int | None = None, filters: dict | None = None) -> list[RetrievedResult]:
#     k = k or cfg("retriever", "k")
#     k_fetch = k * 10
#     norm_filters = {
#         col: [str(v).lower() for v in (val if isinstance(val, list) else [val])] for col, val in (filters or {}).items()
#     }

#     results: list[RetrievedResult] = []
#     seen_parents: set = set()

#     for doc in _search_candidates(query, k_fetch):
#         parent_id = doc.metadata.get("parent_id")
#         if parent_id is None or parent_id in seen_parents:
#             continue
#         seen_parents.add(parent_id)

#         parent_doc = PARENT_STORE[parent_id]
#         records = _build_row_records(parent_doc.metadata)
#         kept = _filter_row_records(records, norm_filters)
#         if not kept:
#             continue

#         list_meta = _records_to_list_meta(kept)
#         hadiths = list_meta.pop("hadith", [])
#         if not hadiths:
#             continue

#         results.append(RetrievedResult(sharh=parent_doc.page_content, hadiths=hadiths, list_meta=list_meta))
#         if len(results) >= k:
#             break

#     return results


# print(f"Retriever ready ✓  (search_type={cfg('retriever', 'search_type')})")

## Query Rewriting

In [15]:
import json as _json

try:
    from groq import Groq as _Groq
except ImportError:
    _Groq = None

_groq_client = None


def _get_groq_client():
    global _groq_client
    if _groq_client is None:
        if _Groq is None:
            raise RuntimeError("groq package not installed — run: pip install groq")
        api_key = _get_api_key("GROQ_API_KEY")
        if not api_key:
            raise RuntimeError("GROQ_API_KEY not set")
        _groq_client = _Groq(api_key=api_key)
    return _groq_client


def _extract_json(raw: str) -> dict:
    cleaned = raw.strip().removeprefix("```json").removesuffix("```").strip()
    return _json.loads(cleaned)


_QUERY_REWRITE_SYSTEM_PROMPT = (
    "حول النص التالي من العامية المصرية إلى العربية الفصحى مع الحفاظ على المعنى فقط، ولا تضف أي معلومات\n"
    "أخرج JSON فقط: {\"msa\": \"...\"}"
)


def preprocess_query_llm(query: str, model: str = "llama-3.3-70b-versatile") -> dict:
    """Return JSON with at least an 'msa' field."""
    client = _get_groq_client()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": _QUERY_REWRITE_SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    try:
        return _extract_json(resp.choices[0].message.content)
    except _json.JSONDecodeError:
        return {"msa": query}

@traceable(name="rewrite_query")
def rewrite_query(question: str) -> str:
    """Rewrite to MSA; falls back to the original text on any error."""
    try:
        return preprocess_query_llm(question).get("msa") or question
    except Exception as e:
        print(f"[rewrite_query] using raw query — {e}")
        return question


print("rewrite_query ready ✓")

rewrite_query ready ✓


## LLM

In [16]:
def build_llm():
    provider    = cfg("llm", "provider")
    temperature = cfg("llm", "temperature")

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=cfg("llm", "openai_model"), temperature=temperature)

    if provider == "ollama":
        from langchain_community.chat_models import ChatOllama
        return ChatOllama(model=cfg("llm", "ollama_model"), temperature=temperature)

    if provider == "groq":
        from langchain_groq import ChatGroq
        api_key = _get_api_key("GROQ_API_KEY")
        return ChatGroq(model=cfg("llm", "groq_model"), temperature=temperature, groq_api_key=api_key)

    if provider == "huggingface":
        from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
        api_key = _get_api_key("HF_TOKEN")
        endpoint = HuggingFaceEndpoint(
            repo_id=cfg("llm", "huggingface_model"),
            temperature=temperature,
            huggingfacehub_api_token=api_key,
            task="text-generation",
            max_new_tokens=CONFIG["llm"].get("max_new_tokens", 512),
        )
        return ChatHuggingFace(llm=endpoint)

    if provider == "huggingface_local":
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
        from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
        model_name = cfg("llm", "huggingface_model")
        tokenizer  = AutoTokenizer.from_pretrained(model_name)
        model      = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, temperature=temperature)
        return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
    if provider == 'fanar':
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
        model="Fanar-C-2-27B",
        temperature=.2,
        api_key="RWZ4TwNuDlgi7Etv0aPUVokcxwXTJYvE",
        base_url="https://api.fanar.qa/v1",
    )
    raise ValueError(f"Unknown llm provider: {provider}")


try:
    llm = build_llm()
    print(f"LLM ready: {cfg('llm', 'provider')}")
except Exception as e:
    print(f"LLM error: {e}")
    llm = None


LLM ready: groq


## Prompt

In [17]:
def format_docs(docs: list[RetrievedResult]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = doc.list_meta
        rawy_list = meta.get("rawy", [])
        hokm_list = meta.get("hokm", [])
        source_list = meta.get("source", [])
        page_id_list = meta.get("page_id", [])

        hadiths_raw = doc.hadiths
        if isinstance(hadiths_raw, str):
            try:
                hadiths_raw = ast.literal_eval(hadiths_raw)
            except Exception:
                hadiths_raw = [hadiths_raw]

        hadith_block = []
        for j, (hadith, rawy, hokm, source, page_id) in enumerate(
            zip(hadiths_raw, rawy_list, hokm_list, source_list, page_id_list), 1
        ):
            hadith_block.append(
                f"""<hadith id="{j}">
            <text>
            {hadith}
            </text>
            <rawy>{rawy}</rawy>
            <source>{source}</source>
            <page_id>{page_id}</page_id>
            <hokm>{hokm}</hokm>
            </hadith>""".strip()
                        )

        blocks.append(
            f"""<document id="{i}">
            <sharh>
            {doc.sharh}
            </sharh>
            <hadiths>
            {"\n\n".join(hadith_block)}
            </hadiths>
            </document>""".strip()
        )

    return "\n\n".join(blocks)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        cfg("prompt", "system_role") + f"\n\nأجب باللغة: {cfg('prompt', 'language')}"
    ),
    ("assistant", "السياق المسترجع:\n{context}\n"),
    ("human", "سؤالي هو:\n{question}"),
])

prompt = RAG_PROMPT
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة\n\nأجب باللغة: ar'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='سؤالي هو:\n{question}'), additional_kwargs={})])

## Chain

In [18]:
rag_chain = (
    {
        "context": RunnableLambda(lambda q: retrieve(q))
                   | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain

{
  context: RunnableLambda(...)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة\n\nأجب باللغة: ar'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='سؤالي هو:\n{question}'), additional_kwargs={})])
| ChatGroq(profile={'max

## Ask

In [19]:
test_docs = retrieve("ماذا يفعل من أتاه خبر يسره ؟", k=5)
test_docs[1].hadiths

['- إنَّ أخوفَ ما أخافُ عليكم الشِّركُ الأصغرُ قالوا يا رسولَ اللهِ وما الشِّركُ الأصغرُ قال الرِّياءُ يُقالُ لمَن يفعَلُ ذلك إذا جاء النَّاسُ بأعمالِهم اذهَبوا إلى الَّذين كنتُم تُراؤون فاطلُبوا ذلك عندَهم',
 '- أخوفُ ما أخافُ عليكم الشركُ الأصغرُ فسئل عنه قال: الرياءُ، يقولُ اللهُ يومَ القيامةِ للمرائين.',
 '- اتقوا الرِّياءَ ، فإنه الشركُ الأصغرُ',
 '- كنا نعُدُّ على عهدِ رسولِ اللهِ صلَّى اللهُ عليهِ وسلَّمَ الشركَ الأصغرَ الرياءَ',
 '- الرياءُ الشركُ الأصغرُ [يعني حديث: إنَّ أخوفَ ما أخافُ عليْكم الشِّرْكَ الأصغرَ قالوا : وما الشِّرْكُ الأصغرُ ؟ قالَ : الرِّياءُ]',
 '- إنَّ أخوَفَ ما أخافُ عليكم الشِّركُ الأصغَرُ؛ الرِّياءُ.',
 '- إنَّ رسولَ اللهِ صلَّى اللهُ عليه وسلَّم قال: إنَّ أخْوَفَ ما أخافُ عليكم... فذَكَرَ مَعْناه. [أي مَعْنى حديثِ: إنَّ أخْوَفَ ما أخافُ عليكم الشِّركُ الأصغَرُ. قالوا: وما الشِّركُ الأصغَرُ يا رسولَ اللهِ؟ قال: الرِّياءُ، يقولُ اللهُ عزَّ وجلَّ لهم يَومَ القيامةِ، إذا جُزِيَ الناسُ بأعْمالِهِم: اذْهَبوا إلى الذين كُنتُم تُراؤونَ في الدُّنيا، فانْظُروا هل ت

In [20]:
test_docs[0].hadiths

['- سَمِعتُ أبا جَمرةَ الضُّبَعيَّ، قال: تَمَتَّعتُ فنَهاني ناسٌ عن ذلك، فأتَيتُ ابنَ عَبَّاسٍ فسَألتُه عن ذلك، فأمَرَني بها، قال: ثُمَّ انطَلَقتُ إلى البَيتِ فنِمتُ، فأتاني آتٍ في مَنامي، فقال: عُمرةٌ مُتَقَبَّلةٌ، وحَجٌّ مَبرورٌ، قال: فأتَيتُ ابنَ عَبَّاسٍ، فأخبَرتُه بالذي رَأيتُ، فقال: اللهُ أكبَرُ اللهُ أكبَرُ، سُنَّةُ أبي القاسِمِ صلَّى اللهُ عليه وسلَّم.',
 '- تَمَتَّعتُ فنَهاني ناسٌ، فسَألتُ ابنَ عَبَّاسٍ رَضيَ اللهُ عنهما، فأمَرَني، فرَأيتُ في المَنامِ كَأنَّ رَجُلًا يقولُ لي: حَجٌّ مَبرورٌ، وعُمرةٌ مُتَقَبَّلةٌ، فأخبَرتُ ابنَ عَبَّاسٍ، فقال: سُنَّةَ النَّبيِّ صلَّى اللهُ عليه وسلَّم، فقال لي: أقِمْ عِندي، فأجعَلَ لكَ سَهمًا مِن مالي. قال شُعبةُ: فقُلتُ: لمَ؟ فقال: للرُّؤيا التي رَأيتُ.']

In [21]:
for i, do in enumerate(test_docs):
    print(f"hadith number {i+1}")
    for hadith in do.hadiths:
        print(hadith, "\n---\n")

hadith number 1
- سَمِعتُ أبا جَمرةَ الضُّبَعيَّ، قال: تَمَتَّعتُ فنَهاني ناسٌ عن ذلك، فأتَيتُ ابنَ عَبَّاسٍ فسَألتُه عن ذلك، فأمَرَني بها، قال: ثُمَّ انطَلَقتُ إلى البَيتِ فنِمتُ، فأتاني آتٍ في مَنامي، فقال: عُمرةٌ مُتَقَبَّلةٌ، وحَجٌّ مَبرورٌ، قال: فأتَيتُ ابنَ عَبَّاسٍ، فأخبَرتُه بالذي رَأيتُ، فقال: اللهُ أكبَرُ اللهُ أكبَرُ، سُنَّةُ أبي القاسِمِ صلَّى اللهُ عليه وسلَّم. 
---

- تَمَتَّعتُ فنَهاني ناسٌ، فسَألتُ ابنَ عَبَّاسٍ رَضيَ اللهُ عنهما، فأمَرَني، فرَأيتُ في المَنامِ كَأنَّ رَجُلًا يقولُ لي: حَجٌّ مَبرورٌ، وعُمرةٌ مُتَقَبَّلةٌ، فأخبَرتُ ابنَ عَبَّاسٍ، فقال: سُنَّةَ النَّبيِّ صلَّى اللهُ عليه وسلَّم، فقال لي: أقِمْ عِندي، فأجعَلَ لكَ سَهمًا مِن مالي. قال شُعبةُ: فقُلتُ: لمَ؟ فقال: للرُّؤيا التي رَأيتُ. 
---

hadith number 2
- إنَّ أخوفَ ما أخافُ عليكم الشِّركُ الأصغرُ قالوا يا رسولَ اللهِ وما الشِّركُ الأصغرُ قال الرِّياءُ يُقالُ لمَن يفعَلُ ذلك إذا جاء النَّاسُ بأعمالِهم اذهَبوا إلى الَّذين كنتُم تُراؤون فاطلُبوا ذلك عندَهم 
---

- أخوفُ ما أخافُ عليكم الشركُ الأصغرُ فسئل عنه 

In [22]:
def ask(question: str, *, show_sources: bool = True, rewrite: bool = True, **filters) -> str:
    active_filters = {**filters}
    # search_query = rewrite_query(question) if rewrite else question
    docs = retrieve(question, k=5)

    # if show_sources:
    #     print("--- Retrieved chunks ---")
    #     for doc in docs:
    #         hadiths = doc.hadiths
    #         if isinstance(hadiths, str):
    #             try:
    #                 hadiths = ast.literal_eval(hadiths)
    #             except Exception:
    #                 hadiths = [hadiths]
    #         for h in hadiths:
    #             print(f"hadith: {h}")
    #         print("\n---------------------------------\n")
    #     if active_filters:
    #         print(f"Active filters: {active_filters}")
    #     print("--- Answer ---")
    if show_sources:
        print("--- Retrieved chunks ---")
        for doc in docs:
            hadiths = doc.hadiths
            if isinstance(hadiths, str):
                try:
                    hadiths = ast.literal_eval(hadiths)
                except Exception:
                    hadiths = [hadiths]
            print(f"[dense={doc.dense_score}  bm25={doc.bm25_score}]")
            for h in hadiths:
                print(f"hadith: {h}")
            print("\n---------------------------------\n")
        if active_filters:
            print(f"Active filters: {active_filters}")
        print("--- Answer ---")

    # if active_filters:
    #     temp_chain = (
    #         {
    #             "context": RunnableLambda(
    #                 lambda inp: retrieve(rewrite_query(inp["question"]), filters=active_filters)  # <-- was **active_filters
    #             ) | RunnableLambda(format_docs),
    #             "question": RunnablePassthrough(),
    #         }
    #         | prompt | llm | StrOutputParser()
    #     )
    #     return temp_chain.invoke({"question": question})

    return rag_chain.invoke(question)


# ── Quick smoke test ──────────────────────────────────────────────────────────
# QUESTION = "ماذا يفعل من أتاه خبر يسره ؟"
# QUESTION = "حكم سقوط ذبابة في الإناء"
QUESTION = "ماذا يفعل من أتاه خبر يسره ؟"

answer = ask(QUESTION)
print(answer)

--- Retrieved chunks ---
[dense=0.5271378755569458  bm25=None]
hadith: - سَمِعتُ أبا جَمرةَ الضُّبَعيَّ، قال: تَمَتَّعتُ فنَهاني ناسٌ عن ذلك، فأتَيتُ ابنَ عَبَّاسٍ فسَألتُه عن ذلك، فأمَرَني بها، قال: ثُمَّ انطَلَقتُ إلى البَيتِ فنِمتُ، فأتاني آتٍ في مَنامي، فقال: عُمرةٌ مُتَقَبَّلةٌ، وحَجٌّ مَبرورٌ، قال: فأتَيتُ ابنَ عَبَّاسٍ، فأخبَرتُه بالذي رَأيتُ، فقال: اللهُ أكبَرُ اللهُ أكبَرُ، سُنَّةُ أبي القاسِمِ صلَّى اللهُ عليه وسلَّم.
hadith: - تَمَتَّعتُ فنَهاني ناسٌ، فسَألتُ ابنَ عَبَّاسٍ رَضيَ اللهُ عنهما، فأمَرَني، فرَأيتُ في المَنامِ كَأنَّ رَجُلًا يقولُ لي: حَجٌّ مَبرورٌ، وعُمرةٌ مُتَقَبَّلةٌ، فأخبَرتُ ابنَ عَبَّاسٍ، فقال: سُنَّةَ النَّبيِّ صلَّى اللهُ عليه وسلَّم، فقال لي: أقِمْ عِندي، فأجعَلَ لكَ سَهمًا مِن مالي. قال شُعبةُ: فقُلتُ: لمَ؟ فقال: للرُّؤيا التي رَأيتُ.

---------------------------------

[dense=0.5023306012153625  bm25=None]
hadith: - إنَّ أخوفَ ما أخافُ عليكم الشِّركُ الأصغرُ قالوا يا رسولَ اللهِ وما الشِّركُ الأصغرُ قال الرِّياءُ يُقالُ لمَن يفعَلُ ذلك إذا جاء النَّاسُ 

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-20b` in organization `org_01kt1j8b7tevh9f1dpwbpyvz00` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 10503, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [25]:
QUESTION = "ماذا يفعل من أتاه خبر يسره ؟"
docs = retrieve(QUESTION, k=5)

print("--- Retrieved chunks ---")
for doc in docs:
    hadiths = doc.hadiths
    if isinstance(hadiths, str):
        try:
            hadiths = ast.literal_eval(hadiths)
        except Exception:
            hadiths = [hadiths]
    for h in hadiths:
        print(f"hadith: {h}")
    print("\n---------------------------------\n")

--- Retrieved chunks ---
hadith: - سَمِعتُ أبا جَمرةَ الضُّبَعيَّ، قال: تَمَتَّعتُ فنَهاني ناسٌ عن ذلك، فأتَيتُ ابنَ عَبَّاسٍ فسَألتُه عن ذلك، فأمَرَني بها، قال: ثُمَّ انطَلَقتُ إلى البَيتِ فنِمتُ، فأتاني آتٍ في مَنامي، فقال: عُمرةٌ مُتَقَبَّلةٌ، وحَجٌّ مَبرورٌ، قال: فأتَيتُ ابنَ عَبَّاسٍ، فأخبَرتُه بالذي رَأيتُ، فقال: اللهُ أكبَرُ اللهُ أكبَرُ، سُنَّةُ أبي القاسِمِ صلَّى اللهُ عليه وسلَّم.
hadith: - تَمَتَّعتُ فنَهاني ناسٌ، فسَألتُ ابنَ عَبَّاسٍ رَضيَ اللهُ عنهما، فأمَرَني، فرَأيتُ في المَنامِ كَأنَّ رَجُلًا يقولُ لي: حَجٌّ مَبرورٌ، وعُمرةٌ مُتَقَبَّلةٌ، فأخبَرتُ ابنَ عَبَّاسٍ، فقال: سُنَّةَ النَّبيِّ صلَّى اللهُ عليه وسلَّم، فقال لي: أقِمْ عِندي، فأجعَلَ لكَ سَهمًا مِن مالي. قال شُعبةُ: فقُلتُ: لمَ؟ فقال: للرُّؤيا التي رَأيتُ.

---------------------------------

hadith: - إنَّ أخوفَ ما أخافُ عليكم الشِّركُ الأصغرُ قالوا يا رسولَ اللهِ وما الشِّركُ الأصغرُ قال الرِّياءُ يُقالُ لمَن يفعَلُ ذلك إذا جاء النَّاسُ بأعمالِهم اذهَبوا إلى الَّذين كنتُم تُراؤون فاطلُبوا ذلك عندَهم
hadith: - أخ

---
# Evaluation (RAG Trials)

Compare pipeline settings using:

1. **Hadith-in-retrieval (HAQA)** — did `Hadith_Matn` appear in retrieved chunks before the LLM?
2. **BLEU / ROUGE** — best-match precision & recall vs ground-truth hadith.
3. **RAGAS** — faithfulness, answer relevancy, context precision/recall.

Results saved under `eval_runs/`.


## Evaluation Config

In [127]:
from datetime import datetime, timezone

EVAL_CONFIG = {
    "haqa_csv": PROJECT_ROOT / "data" / "HAQA.csv",
    "max_samples": None,
    "random_seed": 42,
    "output_dir": PROJECT_ROOT / "eval_runs",
    "run_name": None,
    "retrieval": {
        "enabled": True,
        "min_substring_len": 12,
        "min_token_overlap": 0.45,
        "check_metadata_hadith_1": True,
    },
    "bleu_rouge": {
        "enabled": True,
        "strategy": "best_match",   # "best_match" | "concatenate"
    },
    "ragas": {
        "enabled": True,
        "metrics": ["faithfulness", "answer_relevancy", "context_precision", "context_recall"],
        "ground_truth_column": "Expert_Commentary",
        "max_contexts_for_ragas": None,
    },
}


def _eval_run_dir() -> Path:
    name    = EVAL_CONFIG["run_name"] or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dir = Path(EVAL_CONFIG["output_dir"]) / name
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


EVAL_RUN_DIR = _eval_run_dir()
print(f"Eval outputs -> {EVAL_RUN_DIR}")


Eval outputs -> D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\20260627_221655


## HAQA — Hadith-in-Retrieval

In [128]:
from dataclasses import dataclass as _dc


@_dc
class HadithHitResult:
    hit: bool
    method: str | None
    best_overlap: float
    matched_doc_index: str | None


def hadith_in_retrieved_docs(
    hadith_matn: str,
    all_hadith: list[str],
    *,
    min_substring_len: int | None = None,
    min_token_overlap: float | None = None,
) -> HadithHitResult:
    min_substring_len = min_substring_len or EVAL_CONFIG["retrieval"]["min_substring_len"]
    min_token_overlap = min_token_overlap or EVAL_CONFIG["retrieval"]["min_token_overlap"]

    target = normalize_arabic(hadith_matn)
    if len(target) < 3:
        return HadithHitResult(False, "none", 0.0, None)

    best_overlap, best_hadith = 0.0, ""
    for hadith in all_hadith:
        blob_norm = normalize_arabic(hadith)
        if len(target) >= min_substring_len and target in blob_norm:
            return HadithHitResult(True, "substring", 1.0, blob_norm)
        overlap = sequence_matcher(target, blob_norm)
        if overlap > best_overlap:
            best_overlap, best_hadith = overlap, blob_norm

    if best_overlap >= min_token_overlap:
        return HadithHitResult(True, "token_overlap", best_overlap, best_hadith)
    return HadithHitResult(False, "none", best_overlap, best_hadith)


def load_haqa_eval_frame() -> pd.DataFrame:
    path = EVAL_CONFIG["haqa_csv"]
    df   = pd.read_csv(path)
    df   = df.dropna(subset=["Question_Text", "Hadith_Matn"])
    max_n = EVAL_CONFIG["max_samples"]
    if max_n:
        df = df.sample(n=min(max_n, len(df)), random_state=EVAL_CONFIG["random_seed"])
    return df.reset_index(drop=True)


In [129]:
def run_haqa_retrieval_eval(haqa_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(haqa_df.iterrows(), total=len(haqa_df)):
        question = row["Question_Text"]
        hadith   = row["Hadith_Matn"]
        docs     = retrieve(normalize_arabic(question))
        all_hadiths = [h for r in docs for h in r.hadiths]
        hit = hadith_in_retrieved_docs(hadith, all_hadiths)
        rows.append({
            "Record_Id":          row.get("Record_Id"),
            "Question_Id":        row.get("Question_Id"),
            "Question_Text":      question,
            "Hadith_Matn":        hadith,
            "hadith_hit":         hit.hit,
            "hit_method":         hit.method,
            "best_token_overlap": round(hit.best_overlap, 4),
            "n_retrieved":        len(docs),
            "hadith":             hit.matched_doc_index,
        })
    return pd.DataFrame(rows)


if EVAL_CONFIG["retrieval"]["enabled"]:
    haqa_eval_df       = load_haqa_eval_frame()
    retrieval_results_df = run_haqa_retrieval_eval(haqa_eval_df)

    hit_rate = retrieval_results_df["hadith_hit"].mean()
    print(f"Hadith-in-retrieval hit rate: {hit_rate:.1%} ({retrieval_results_df['hadith_hit'].sum()}/{len(retrieval_results_df)})")
    print(retrieval_results_df["hit_method"].value_counts(dropna=False))

    out_path = EVAL_RUN_DIR / "haqa_retrieval_hits.csv"
    retrieval_results_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved -> {out_path}")
    retrieval_results_df.head()
else:
    print("Retrieval eval disabled in EVAL_CONFIG")


100%|██████████| 1597/1597 [04:49<00:00,  5.53it/s]

Hadith-in-retrieval hit rate: 53.9% (860/1597)
hit_method
token_overlap    848
none             737
substring         12
Name: count, dtype: int64
Saved -> D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\20260627_221655\haqa_retrieval_hits.csv


## BLEU & ROUGE Evaluation (best_match strategy)

**Strategy:** compare ground-truth `Hadith_Matn` against every retrieved hadith individually and report the **maximum** BLEU-1 precision / ROUGE-1 recall.  
This is correct when a chunk contains many hadiths and only one matches the ground truth.


In [130]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


def rouge1_recall(reference: str, hypothesis: str) -> float:
    ref_tokens = normalize_arabic(reference).split()
    hyp_tokens = set(normalize_arabic(hypothesis).split())
    if not ref_tokens:
        return 0.0
    return sum(1 for t in ref_tokens if t in hyp_tokens) / len(ref_tokens)


def bleu1_precision(reference: str, hypothesis: str) -> float:
    ref_tokens = set(normalize_arabic(reference).split())
    hyp_tokens = normalize_arabic(hypothesis).split()
    if not hyp_tokens:
        return 0.0
    return sum(1 for t in hyp_tokens if t in ref_tokens) / len(hyp_tokens)


def compute_best_match_scores(ground_truth_hadith: str, retrieved_hadiths: list[str]) -> dict[str, float]:
    if not retrieved_hadiths:
        return {"bleu": 0.0, "rouge1": 0.0}
    best = {"bleu": 0.0, "rouge1": 0.0}
    for hyp in retrieved_hadiths:
        prec = bleu1_precision(ground_truth_hadith, hyp)
        rec  = rouge1_recall(ground_truth_hadith, hyp)
        if prec  > best["bleu"]:   best["bleu"]   = prec
        if rec   > best["rouge1"]: best["rouge1"] = rec
    return best


print("BLEU / ROUGE helpers loaded ✓")


BLEU / ROUGE helpers loaded ✓


In [131]:
def run_bleu_rouge_eval(haqa_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(haqa_df.iterrows(), total=len(haqa_df), desc="BLEU/ROUGE Eval"):
        question     = row["Question_Text"]
        ground_truth = row["Hadith_Matn"]
        docs         = retrieve_filtered(normalize_arabic(question))
        retrieved_hadiths = [normalize_arabic(str(h)) for r in docs for h in r.hadiths]
        gt_norm      = normalize_arabic(ground_truth)
        scores       = compute_best_match_scores(gt_norm, retrieved_hadiths)
        rows.append({
            "Record_Id":           row.get("Record_Id"),
            "Question_Id":         row.get("Question_Id"),
            "Question_Text":       question,
            "Hadith_Matn":         ground_truth,
            "n_retrieved_hadiths": len(retrieved_hadiths),
            "precision_bleu":      round(scores["bleu"],   4),
            "recall_rouge1":       round(scores["rouge1"], 4),
        })
    return pd.DataFrame(rows)


if EVAL_CONFIG["bleu_rouge"]["enabled"]:
    if "haqa_eval_df" not in globals():
        haqa_eval_df = load_haqa_eval_frame()

    bleu_rouge_df = run_bleu_rouge_eval(haqa_eval_df)
    print("=" * 55)
    print("BLEU / ROUGE Summary (Best Match)")
    print(f"  Precision (BLEU-1):  {bleu_rouge_df['precision_bleu'].mean():.4f}")
    print(f"  Recall    (ROUGE-1): {bleu_rouge_df['recall_rouge1'].mean():.4f}")
    print("=" * 55)

    out_path = EVAL_RUN_DIR / "haqa_bleu_rouge.csv"
    bleu_rouge_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved -> {out_path}")
    bleu_rouge_df.head()
else:
    print("BLEU/ROUGE eval disabled in EVAL_CONFIG")


BLEU/ROUGE Eval: 100%|██████████| 1597/1597 [03:54<00:00,  6.80it/s]

BLEU / ROUGE Summary (Best Match)
  Precision (BLEU-1):  0.4896
  Recall    (ROUGE-1): 0.5087
Saved -> D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\20260627_221655\haqa_bleu_rouge.csv


## RAGAS

In [132]:
!pip install ragas --quiet

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [133]:
import json

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness

RAGAS_METRIC_MAP = {
    "faithfulness":       faithfulness,
    "answer_relevancy":   answer_relevancy,
    "context_precision":  context_precision,
    "context_recall":     context_recall,
}


def _ground_truth_from_row(row: pd.Series) -> str:
    col = EVAL_CONFIG["ragas"]["ground_truth_column"]
    if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
        return str(row[col])
    if pd.notna(row.get("Full_Answer")):
        return str(row["Full_Answer"])
    return str(row.get("Expert_Commentary", ""))


def build_ragas_dataset(haqa_df: pd.DataFrame) -> Dataset:
    rows = []
    max_ctx = EVAL_CONFIG["ragas"]["max_contexts_for_ragas"]
    for _, row in haqa_df.iterrows():
        question = row["Question_Text"]
        docs     = retrieve(question)
        contexts = [d.page_content for d in docs]
        if max_ctx:
            contexts = contexts[:max_ctx]
        answer = rag_chain.invoke({"question": question})
        rows.append({
            "question":     question,
            "answer":       answer,
            "contexts":     contexts,
            "ground_truth": _ground_truth_from_row(row),
            "hadith_matn":  row["Hadith_Matn"],
        })
    return Dataset.from_list(rows)


def run_ragas_eval(haqa_df: pd.DataFrame):
    metrics = [RAGAS_METRIC_MAP[m] for m in EVAL_CONFIG["ragas"]["metrics"]]
    dataset = build_ragas_dataset(haqa_df)
    return evaluate(dataset=dataset, metrics=metrics, llm=llm, embeddings=embeddings)


if EVAL_CONFIG["ragas"]["enabled"]:
    if "haqa_eval_df" not in globals():
        haqa_eval_df = load_haqa_eval_frame()

    ragas_result         = run_ragas_eval(haqa_eval_df)
    ragas_per_question_df = ragas_result.to_pandas()

    per_row_path = EVAL_RUN_DIR / "ragas_per_question.csv"
    ragas_per_question_df.to_csv(per_row_path, index=False, encoding="utf-8-sig")

    metric_cols = [c for c in ragas_per_question_df.columns if c in EVAL_CONFIG["ragas"]["metrics"]]
    ragas_mean  = ragas_per_question_df[metric_cols].mean(numeric_only=True).to_dict()

    summary: dict = {
        "run_dir":        str(EVAL_RUN_DIR),
        "n_samples":      len(haqa_eval_df),
        "config_snapshot": CONFIG,
        "eval_config":    {k: v for k, v in EVAL_CONFIG.items() if k != "haqa_csv"},
        "ragas_mean":     ragas_mean,
    }
    if "retrieval_results_df" in globals():
        summary["hadith_hit_rate"] = float(retrieval_results_df["hadith_hit"].mean())
    if "bleu_rouge_df" in globals():
        summary["bleu_rouge_mean"] = {
            col: round(float(bleu_rouge_df[col].mean()), 4)
            for col in ["precision_bleu", "recall_rouge1"]
            if col in bleu_rouge_df.columns
        }

    summary_path = EVAL_RUN_DIR / "run_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    print("RAGAS (mean over samples):")
    for k, v in ragas_mean.items():
        print(f"  {k}: {v:.4f}")
    print(f"Saved -> {per_row_path}, {summary_path}")
    ragas_per_question_df.head()
else:
    print("RAGAS eval disabled in EVAL_CONFIG")


C:\Users\moham\AppData\Local\Temp\ipykernel_8472\3443294392.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
C:\Users\moham\AppData\Local\Temp\ipykernel_8472\3443294392.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
C:\Users\moham\AppData\Local\Temp\ipykernel_8472\3443294392.py:5: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: 

AttributeError: 'RetrievedResult' object has no attribute 'page_content'

## Compare Runs

In [ ]:
def load_eval_summaries(runs_root: Path | None = None) -> pd.DataFrame:
    runs_root = runs_root or Path(EVAL_CONFIG["output_dir"])
    rows = []
    for p in sorted(runs_root.glob("*/run_summary.json")):
        data = json.loads(p.read_text(encoding="utf-8"))
        row  = {"run": p.parent.name}
        row.update(data.get("ragas_mean", {}))
        row["hadith_hit_rate"] = data.get("hadith_hit_rate")
        row.update(data.get("bleu_rouge_mean", {}))
        rows.append(row)
    return pd.DataFrame(rows)


# Uncomment after 2+ eval runs:
# compare_df = load_eval_summaries()
# compare_df.sort_values("hadith_hit_rate", ascending=False)
